# 4.2 Time Series Modeling & Comparative Analysis

**Metro Interstate Traffic Volume (I-94 Westbound)**

## Motivation & Objectives
In `04.1_modeling_trial.ipynb`, we built a **Tabular ML model** (XGBoost) that maps calendar and weather features at time $t$ to traffic volume:
$$\hat{y}_t = f(\text{calendar}_t, \text{weather}_t)$$

While effective ($R^2 = 0.963$), the tabular model treats each hour as an independent row and ignores **temporal momentum** (what happened in the last 1–2 hours, or at the exact same hour yesterday and last week).

In this notebook, we explore **Time Series and AutoRegressive Modeling**:
1. **Analyze Autocorrelation & Seasonality:** ACF, PACF, and seasonal decomposition.
2. **Classical Statistical Baseline:** SARIMAX with exogenous weather features.
3. **AutoRegressive Lag-Engineered Machine Learning:** XGBoost with $y_{t-1}, y_{t-2}, y_{t-24}, y_{t-168}$ and rolling statistics.
4. **Direct Head-to-Head Comparison:** Compare metrics against Notebook 04.1 Tabular XGBoost.
5. **Production Model Export:** Save the serialized lag pipeline for hybrid deployment.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor
import joblib
import json
from pathlib import Path

## 1. Time Series Setup & Regularization

We load the cleaned dataset, set `date_time` as a datetime index, and reindex to an unbroken regular 1-hour grid (`freq='1h'`) to ensure that time differences between lags correspond to exact hours.

In [2]:
DATA_PATH = Path('../data/processed/traffic_volume_cleaned.csv')
df = pd.read_csv(DATA_PATH)
df['date_time'] = pd.to_datetime(df['date_time'])
df = df.sort_values('date_time').reset_index(drop=True)

- Resample to a continuous hourly frequency

In [3]:
ts_df = df.set_index('date_time').resample('1h').asfreq()

- Impute continuous series for lag generation

In [4]:
ts_df['traffic_volume_filled'] = ts_df['traffic_volume'].interpolate(method='time')
ts_df['temp'] = ts_df['temp'].interpolate(method='time')
ts_df['rain_1h'] = ts_df['rain_1h'].fillna(0)
ts_df['snow_1h'] = ts_df['snow_1h'].fillna(0)
ts_df['clouds_all'] = ts_df['clouds_all'].interpolate(method='time')
ts_df['weather_main'] = ts_df['weather_main'].ffill().bfill()
ts_df['holiday'] = ts_df['holiday'].fillna('Not Holiday')

- Extract temporal features

In [5]:
ts_df['hour'] = ts_df.index.hour
ts_df['day_of_week'] = ts_df.index.day_name()
ts_df['month'] = ts_df.index.month
ts_df['hour_sin'] = np.sin(2 * np.pi * ts_df['hour'] / 24)
ts_df['hour_cos'] = np.cos(2 * np.pi * ts_df['hour'] / 24)
ts_df['month_sin'] = np.sin(2 * np.pi * ts_df['month'] / 12)
ts_df['month_cos'] = np.cos(2 * np.pi * ts_df['month'] / 12)

print(f'Total regularized timeline hours: {len(ts_df)}')
print(f'Original recorded observations  : {len(df)}')

Total regularized timeline hours: 52551
Original recorded observations  : 40564


## 2. Autocorrelation & Seasonality Analysis (ACF / PACF)

We examine the Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) to identify key temporal dependencies:

In [6]:
series = ts_df['traffic_volume_filled'].dropna()
acf_vals = acf(series, nlags=168, fft=True)
pacf_vals = pacf(series, nlags=48)

print(f'Lag 1 Autocorrelation (Immediate 1h momentum) : {acf_vals[1]:.4f}')
print(f'Lag 2 Autocorrelation (2h momentum)           : {acf_vals[2]:.4f}')
print(f'Lag 24 Autocorrelation (Daily commute cycle)   : {acf_vals[24]:.4f}')
print(f'Lag 168 Autocorrelation (Same hour last week)  : {acf_vals[168]:.4f}')

Lag 1 Autocorrelation (Immediate 1h momentum) : 0.9259
Lag 2 Autocorrelation (2h momentum)           : 0.7643
Lag 24 Autocorrelation (Daily commute cycle)   : 0.8594
Lag 168 Autocorrelation (Same hour last week)  : 0.9080


## 3. Autoregressive Feature Engineering

Based on the ACF/PACF analysis, we construct:
- **`lag_1`, `lag_2`:** Short-term inertia / momentum.
- **`lag_24`:** Daily seasonal baseline (same hour yesterday).
- **`lag_168`:** Weekly seasonal baseline (same hour last week).
- **`rolling_mean_6h`, `rolling_mean_24h`:** Moving average trends over 6h and 24h windows.

In [7]:
ts_df['lag_1'] = ts_df['traffic_volume_filled'].shift(1)
ts_df['lag_2'] = ts_df['traffic_volume_filled'].shift(2)
ts_df['lag_24'] = ts_df['traffic_volume_filled'].shift(24)
ts_df['lag_168'] = ts_df['traffic_volume_filled'].shift(168)
ts_df['rolling_mean_6h'] = ts_df['traffic_volume_filled'].shift(1).rolling(6).mean()
ts_df['rolling_mean_24h'] = ts_df['traffic_volume_filled'].shift(1).rolling(24).mean()

- Evaluate only on observed ground-truth records where lags are valid

In [8]:
eval_df = ts_df[ts_df['traffic_volume'].notna() & ts_df['lag_168'].notna()].copy()

- Chronological 80/20 train/test split matching Notebook 04.1

In [9]:
split_idx = int(len(eval_df) * 0.8)
train_df = eval_df.iloc[:split_idx]
test_df = eval_df.iloc[split_idx:]

print(f'Train set: {len(train_df)} rows ({train_df.index.min()} to {train_df.index.max()})')
print(f'Test set : {len(test_df)} rows ({test_df.index.min()} to {test_df.index.max()})')

Train set: 32324 rows (2012-10-09 09:00:00 to 2017-10-28 02:00:00)
Test set : 8082 rows (2017-10-28 03:00:00 to 2018-09-30 23:00:00)


## 4. Model Training: Classical SARIMAX vs. AutoRegressive XGBoost

In [10]:
num_features = [
    'temp', 'rain_1h', 'snow_1h', 'clouds_all',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'lag_1', 'lag_2', 'lag_24', 'lag_168',
    'rolling_mean_6h', 'rolling_mean_24h'
]
cat_features = ['holiday', 'weather_main', 'day_of_week']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

lag_xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1))
])

X_train = train_df[num_features + cat_features]
y_train = train_df['traffic_volume']
X_test = test_df[num_features + cat_features]
y_test = test_df['traffic_volume']



- Train Lag-XGBoost

In [11]:
lag_xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = lag_xgb_pipeline.predict(X_test)

xgb_mae = mean_absolute_error(y_test, y_pred_xgb)
xgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
xgb_r2 = r2_score(y_test, y_pred_xgb)



C:\Users\Abdo\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


- Train SARIMAX Baseline (on representative subsample for demonstration)

In [12]:
sarimax_sample = train_df['traffic_volume'].iloc[-1000:].values
sarimax_exog = train_df[['temp', 'rain_1h', 'hour_sin', 'hour_cos']].iloc[-1000:].values
sarimax_model = SARIMAX(sarimax_sample, exog=sarimax_exog, order=(1, 0, 1), seasonal_order=(1, 0, 0, 24))
sarimax_res = sarimax_model.fit(disp=False)


- SARIMAX test prediction

In [13]:
test_exog = test_df[['temp', 'rain_1h', 'hour_sin', 'hour_cos']].iloc[:200].values
sarimax_pred = sarimax_res.forecast(steps=200, exog=test_exog)
sarimax_mae = mean_absolute_error(y_test.iloc[:200], sarimax_pred)
sarimax_rmse = np.sqrt(mean_squared_error(y_test.iloc[:200], sarimax_pred))
sarimax_r2 = r2_score(y_test.iloc[:200], sarimax_pred)


## 5. Comparative Benchmark: Tabular vs. Time Series Models

We compare the results directly against Notebook 04.1 Tabular XGBoost:

In [14]:
benchmark_results = pd.DataFrame([
    {
        'Model': 'AutoRegressive Lag-XGBoost (Notebook 04.2)',
        'MAE (Veh/hr)': round(xgb_mae, 2),
        'RMSE (Veh/hr)': round(xgb_rmse, 2),
        'R2 Score': round(xgb_r2, 4),
        'Approach': 'Time Series / Lag-Enhanced',
        'Requires Live Sensor?': 'Yes (yt-1, yt-24, rolling stats)'
    },
    {
        'Model': 'Tabular XGBoost (Notebook 04.1)',
        'MAE (Veh/hr)': 234.07,
        'RMSE (Veh/hr)': 379.52,
        'R2 Score': 0.9629,
        'Approach': 'Tabular (Calendar + Weather)',
        'Requires Live Sensor?': 'No (Weather API only)'
    },
    {
        'Model': 'SARIMAX Statistical Baseline',
        'MAE (Veh/hr)': round(sarimax_mae, 2),
        'RMSE (Veh/hr)': round(sarimax_rmse, 2),
        'R2 Score': round(sarimax_r2, 4),
        'Approach': 'Classical Statistical Time Series',
        'Requires Live Sensor?': 'Yes'
    }
])

print('=== Benchmark Comparison ===')
print(benchmark_results.to_string(index=False))

=== Benchmark Comparison ===
                                     Model  MAE (Veh/hr)  RMSE (Veh/hr)  R2 Score                          Approach            Requires Live Sensor?
AutoRegressive Lag-XGBoost (Notebook 04.2)        146.71         229.60    0.9864        Time Series / Lag-Enhanced Yes (yt-1, yt-24, rolling stats)
           Tabular XGBoost (Notebook 04.1)        234.07         379.52    0.9629      Tabular (Calendar + Weather)            No (Weather API only)
              SARIMAX Statistical Baseline        892.34        1172.95    0.6595 Classical Statistical Time Series                              Yes


## 6. Save Lag-Engineered Pipeline to `src/models/`

We export the trained Lag-XGBoost pipeline and metadata so it can be utilized in our production Hybrid Architecture.

In [15]:
MODEL_DIR = Path('../src/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

lag_pipeline_path = MODEL_DIR / 'traffic_volume_lag_pipeline.joblib'
joblib.dump(lag_xgb_pipeline, lag_pipeline_path)

lag_metadata = {
    'model_name': 'AutoRegressive Lag-XGBoost',
    'numerical_features': num_features,
    'categorical_features': cat_features,
    'test_mae': round(float(xgb_mae), 2),
    'test_rmse': round(float(xgb_rmse), 2),
    'test_r2': round(float(xgb_r2), 4),
    'pipeline_path': str(lag_pipeline_path)
}

lag_metadata_path = MODEL_DIR / 'traffic_volume_lag_metadata.json'
lag_metadata_path.write_text(json.dumps(lag_metadata, indent=2), encoding='utf-8')

print(f'[OK] Lag Pipeline saved : {lag_pipeline_path}')
print(f'[OK] Lag Metadata saved : {lag_metadata_path}')

[OK] Lag Pipeline saved : ..\src\models\traffic_volume_lag_pipeline.joblib
[OK] Lag Metadata saved : ..\src\models\traffic_volume_lag_metadata.json


## 7. Conclusions & Hybrid Production Strategy

### Core Takeaways:
1. **Accuracy Gain:** AutoRegressive Lag-XGBoost reduces MAE from **234.07 down to ~146.63 veh/hr** and improves $R^2$ to **0.9864**, because immediate traffic inertia ($y_{t-1}, y_{t-2}$) and daily cycles ($y_{t-24}$) account for sudden momentum shifts that weather and calendar alone cannot capture.
2. **Operational Constraint:** Lag models require **continuous real-time traffic sensor inputs**. If a sensor is offline or a user wants to forecast 3 days into the future, lags must be estimated recursively, which accumulates error.
3. **The Hybrid Solution:** In production:
   - **Near-term / Live Momentum (Next 1–4 hours):** Use **Lag-XGBoost** when recent sensor readings are provided.
   - **Medium / Multi-Day Forecast (1–3 days ahead):** Seamlessly fall back to **Tabular XGBoost** (Notebook 04.1), which requires only a weather forecast and remains rock-solid without compounding error.